In [ ]:
# In[1]:

import pandas as pd
import pytz

# timezone for UTC+8
tz = pytz.timezone('Asia/Shanghai')

# Load CSVs (they are in current working directory)
metric_df = pd.read_csv('metric.csv')
trace_df = pd.read_csv('trace.csv')
log_df = pd.read_csv('log.csv')
error_df = pd.read_csv('error_logs.csv')

# Helper to produce range summary (raw ts and converted to UTC+8 strings)
def ts_range_summary(df, ts_col='timestamp'):
    min_ts = int(df[ts_col].min()) if not df.empty else None
    max_ts = int(df[ts_col].max()) if not df.empty else None
    if min_ts is not None:
        min_dt = pd.to_datetime(min_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
        max_dt = pd.to_datetime(max_ts, unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z')
    else:
        min_dt = max_dt = None
    return pd.DataFrame([{'min_ts': min_ts, 'max_ts': max_ts, 'min_dt_UTC+8': min_dt, 'max_dt_UTC+8': max_dt}])

# 1) Overall min/max for each file
metric_range = ts_range_summary(metric_df)
trace_range = ts_range_summary(trace_df)
log_range = ts_range_summary(log_df)
error_range = ts_range_summary(error_df)

# 2) metric.csv: unique cmdb_id values (sorted) and top 20 (cmdb_id, kpi_name, count)
metric_cmdbs = pd.DataFrame({'cmdb_id': sorted(metric_df['cmdb_id'].dropna().unique())})
metric_top20 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 3) trace.csv: unique cmdb_id values (sorted) and top 20 (cmdb_id, trace_name, count)
trace_cmdbs = pd.DataFrame({'cmdb_id': sorted(trace_df['cmdb_id'].dropna().unique())})
trace_top20 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 4) log.csv: unique cmdb_id values (sorted) and top 20 (cmdb_id, log_name, count)
log_cmdbs = pd.DataFrame({'cmdb_id': sorted(log_df['cmdb_id'].dropna().unique())})
log_top20 = (
    log_df
    .groupby(['cmdb_id', 'log_name'], dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5) error_logs.csv: unique cmdb_id values (sorted), total row count, top 20 (cmdb_id, count) and earliest/latest ts for top 5 cmdb_ids
error_cmdbs = pd.DataFrame({'cmdb_id': sorted(error_df['cmdb_id'].dropna().unique())})
error_total_rows = int(len(error_df))

error_top20 = (
    error_df
    .groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# earliest and latest timestamp per cmdb_id for top 5 cmdb_ids
top5_cmdbs = error_top20['cmdb_id'].head(5).tolist()
if top5_cmdbs:
    error_top5_ts = (
        error_df[error_df['cmdb_id'].isin(top5_cmdbs)]
        .groupby('cmdb_id', dropna=False)['timestamp']
        .agg(min_ts='min', max_ts='max')
        .reset_index()
    )
    # add converted UTC+8 datetime strings
    error_top5_ts['min_dt_UTC+8'] = error_top5_ts['min_ts'].apply(lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))
    error_top5_ts['max_dt_UTC+8'] = error_top5_ts['max_ts'].apply(lambda t: pd.to_datetime(int(t), unit='s', utc=True).tz_convert(tz).strftime('%Y-%m-%d %H:%M:%S %Z'))
    error_top5_ts = error_top5_ts[['cmdb_id', 'min_ts', 'min_dt_UTC+8', 'max_ts', 'max_dt_UTC+8']]
else:
    error_top5_ts = pd.DataFrame(columns=['cmdb_id','min_ts','min_dt_UTC+8','max_ts','max_dt_UTC+8'])

# Final compact outputs (variables displayed, not printed)
metric_range, metric_cmdbs, metric_top20, trace_range, trace_cmdbs, trace_top20, log_range, log_cmdbs, log_top20, error_range, error_cmdbs, error_total_rows, error_top20, error_top5_ts

```
Out[1]:
```
summary = (
    "Summary of telemetry files (timestamps reported in UTC+8 where converted):\n\n"
    "metric.csv:\n"
    "- time range: min_ts=1615532400, max_ts=1615534200 "
    "(2021-03-12 15:00:00 to 2021-03-12 15:30:00 UTC+8)\n"
    "- unique cmdb_id count: 29 (examples: IG01, IG02, MG01, MG02, Mysql01, Tomcat04, apache02, dockerA1, dockerA2, dockerB1, dockerB2)\n"
    "- top combinations (cmdb_id, kpi_name, count): top counts are 31 for many entries. Top examples: \n"
    "  * (IG01, JVM-Threads_7778_JVM_ThreadCount_Threads, 31)\n"
    "  * (apache02, OSLinux-OSLinux_ZABBIX_Host_Uptime, 31)\n"
    "  * (IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax, 31)\n"
    "  * (Mysql01, OSLinux-OSLinux_ZABBIX_Host_Uptime, 31)\n"
    "  * multiple Tomcat04 session/thread/request KPIs at count=31\n\n"
    "trace.csv:\n"
    "- time range: min_ts=1615532400, max_ts=1615534200 "
    "(2021-03-12 15:00:00 to 2021-03-12 15:30:00 UTC+8)\n"
    "- unique cmdb_id count: 12 (examples: IG01, IG02, Tomcat02, Tomcat03, Tomcat04, dockerA1, dockerA2, dockerB2, etc.)\n"
    "- top combinations (cmdb_id, trace_name, count): many entries at count=31. Top examples:\n"
    "  * (Tomcat02, trace.self.row_count, 31)\n"
    "  * (Tomcat03, trace.from_IG02.row_count, 31)\n"
    "  * (Tomcat03, trace.self.duration_mean, 31)\n"
    "  * (IG02, trace.to_Tomcat03.duration_mean, 31)\n"
    "  * dockerB2 trace metrics also at count=31\n"
    "  (Some components show counts of 30 for a few trace metrics, e.g., IG01 and dockerA2)\n\n"
    "log.csv:\n"
    "- time range: min_ts=1615532400, max_ts=1615534200 "
    "(2021-03-12 15:00:00 to 2021-03-12 15:30:00 UTC+8)\n"
    "- unique cmdb_id count: 7 (Mysql01, Tomcat01, Tomcat02, Tomcat03, Tomcat04, apache01, apache02)\n"
    "- top combinations (cmdb_id, log_name, count): many log.row_count and log.error_count at count=31. Top examples:\n"
    "  * (Tomcat01, log.row_count, 31)\n"
    "  * (Tomcat01, log.error_count, 31)\n"
    "  * (apache02, log.error_count, 31)\n"
    "  * (apache01, log.row_count, 31)\n"
    "  * several gc_pause_max entries have lower counts (e.g., 30, 28, 26, 25)\n\n"
    "error_logs.csv:\n"
    "- file is empty in the dataset: min_ts and max_ts are None, unique cmdb_id list is empty\n"
    "- total rows = 0; no per-cmdb error counts or earliest/latest timestamps to report\n\n"
    "Overall note: metric/trace/log data cover the same 30-minute window (2021-03-12 15:00–15:30 UTC+8). "
    "Top counts are mostly 31 (full coverage per-minute across the window), with a few trace/log KPIs at 30 or lower. "
    "error_logs.csv contains no entries."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(       min_ts      max_ts             min_dt_UTC+8             max_dt_UTC+8
0  1615532400  1615534200  2021-03-12 15:00:00 CST  2021-03-12 15:30:00 CST,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4    Mysql01
..       ...
24  apache02
25  dockerA1
26  dockerA2
27  dockerB1
28  dockerB2

[29 rows x 1 columns],      cmdb_id                                           kpi_name  count
0       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31
1   apache02                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
2       IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
3    Mysql01                 OSLinux-OSLinux_ZABBIX_Host_Uptime     31
4   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31
5   Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...     31
6   Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...     31
7   Tomcat04    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter     31
8   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...     31
9   Tomcat04  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...     31
10  Tomcat04        Tomcat-Sessions_7441--_SESSIONActiveCounter     31
11  Tomcat04     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter     31
12  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
13  Tomcat04  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...     31
14  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
15  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
16  Tomcat04  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...     31
17  Tomcat04  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...     31
18      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax     31
19      IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed     31,        min_ts      max_ts             min_dt_UTC+8             max_dt_UTC+8
0  1615532400  1615534200  2021-03-12 15:00:00 CST  2021-03-12 15:30:00 CST,      cmdb_id
0       IG01
1       IG02
2       MG01
3       MG02
4   Tomcat01
5   Tomcat02
6   Tomcat03
7   Tomcat04
8   dockerA1
9   dockerA2
10  dockerB1
11  dockerB2,      cmdb_id                       trace_name  count
0   Tomcat02             trace.self.row_count     31
1   Tomcat03        trace.from_IG02.row_count     31
2   Tomcat03     trace.from_IG02.duration_p95     31
3   Tomcat03    trace.from_IG02.duration_mean     31
4       IG02  trace.to_Tomcat03.duration_mean     31
5   Tomcat03         trace.self.duration_mean     31
6   Tomcat03             trace.self.row_count     31
7   dockerB2         trace.self.duration_mean     31
8   dockerB2          trace.self.duration_p95     31
9   dockerB2             trace.self.row_count     31
10      IG02      trace.to_Tomcat03.row_count     31
11      IG02   trace.to_Tomcat03.duration_p95     31
12  Tomcat02          trace.self.duration_p95     31
13  Tomcat03          trace.self.duration_p95     31
14  Tomcat02         trace.self.duration_mean     31
15      IG01         trace.self.duration_mean     30
16  dockerA2         trace.self.duration_mean     30
17  dockerA2          trace.self.duration_p95     30
18  dockerA1       trace.to_MG01.duration_p95     30
19  dockerA1          trace.to_MG01.row_count     30,        min_ts      max_ts             min_dt_UTC+8             max_dt_UTC+8
0  1615532400  1615534200  2021-03-12 15:00:00 CST  2021-03-12 15:30:00 CST,     cmdb_id
0   Mysql01
1  Tomcat01
2  Tomcat02
3  Tomcat03
4  Tomcat04
5  apache01
6  apache02,      cmdb_id          log_name  count
0   Tomcat01     log.row_count     31
1   Tomcat01   log.error_count     31
2   apache02   log.error_count     31
3   apache01     log.row_count     31
4   Tomcat02   log.error_count     31
5   Tomcat02     log.row_count     31
6   Tomcat03   log.error_count     31
7   Tomcat04     log.row_count     31
8   Tomcat04   log.error_count     31
9   Tomcat03     log.row_count     31
10  apache02     log.row_count     31
11  apache01   log.error_count     31
12  Tomcat02  log.gc_pause_max     30
13   Mysql01   log.error_count     28
14   Mysql01     log.row_count     28
15  Tomcat01  log.gc_pause_max     26
16  Tomcat04  log.gc_pause_max     26
17  Tomcat03  log.gc_pause_max     25,   min_ts max_ts min_dt_UTC+8 max_dt_UTC+8
0   None   None         None         None, Empty DataFrame
Columns: [cmdb_id]
Index: [], 0, Empty DataFrame
Columns: [cmdb_id, count]
Index: [], Empty DataFrame
Columns: [cmdb_id, min_ts, min_dt_UTC+8, max_ts, max_dt_UTC+8]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# reuse existing metric_df loaded earlier
import pandas as pd
import pytz

# Allowed root-cause components
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Ensure value is numeric
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Filter metric rows to allowed components only
metric_allowed = metric_df[metric_df['cmdb_id'].isin(allowed)].copy()

# Compute global aggregates per (cmdb_id, kpi_name)
metric_agg_all = (
    metric_allowed
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p95=lambda x: x.quantile(0.95) if x.notna().any() else float('nan'),
        p5=lambda x: x.quantile(0.05) if x.notna().any() else float('nan')
    )
    .reset_index()
)

# Sort by count descending and keep top 20 for compact output
metric_agg_top20 = metric_agg_all.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Display only the compact top-20 aggregate table
metric_agg_top20

```
Out[2]:
```
summary = (
    "Summary of metric.csv aggregates (allowed components only):\n\n"
    "- Action: computed global aggregates per (cmdb_id, kpi_name) over the full series (count, min, max, mean, p95, p5).\n"
    "- Storage: the full per-(cmdb_id,kpi_name) aggregated table is saved in the kernel variable 'metric_agg_all'.\n\n"
    "Top 20 (cmdb_id, kpi_name) pairs (sorted by count desc):\n"
    "- All top-20 rows have count = 31 (full coverage across the available window).\n"
    "- Dominant component: Redis01 (many Redis metrics appear in the top-20, e.g., used_memory, used_memory_peak, used_memory_rss, total_commands_processed, total_connections_received, uptime_in_seconds, keyspace_misses, lru_clock, maxmemory, pubsub_channels).\n"
    "- Other notable entries in the top-20:\n"
    "  * apache02: OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount and OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT\n"
    "  * IG01: JVM-Threads_7778_JVM_ThreadCount_Threads and JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed\n"
    "  * IG02: OSLinux-CPU_CPU_CPUWio\n\n"
    "Columns included in the output: cmdb_id, kpi_name, count, min, max, mean, p95, p5.\n\n"
    "Conclusion: Redis01 contributes the majority of top metric series by count; several JVM and OS metrics from IG01/IG02 and apache02 also appear among the top-20. The full aggregated table for further analysis is available as 'metric_agg_all'."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  count           min           max          mean           p95            p5
0   apache02   OSLinux-OSLinux_PROCESS_apache_10001_PROCPPCount     31  1.000000e+01  1.000000e+01  1.000000e+01  1.000000e+01  1.000000e+01
1       IG01           JVM-Threads_7778_JVM_ThreadCount_Threads     31  6.700000e+01  7.600000e+01  6.967742e+01  7.450000e+01  6.700000e+01
2   apache02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...     31  4.000000e-01  4.000000e-01  4.000000e-01  4.000000e-01  4.000000e-01
3       IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed     31  5.784948e+07  5.785396e+07  5.785338e+07  5.785396e+07  5.784948e+07
4       IG02                             OSLinux-CPU_CPU_CPUWio     31  2.080000e-02  1.612800e+00  2.011513e-01  8.355515e-01  2.704800e-02
5    Redis01              redis-Redis_6379_Redis  (used_memory)     31  1.762692e+07  1.860906e+07  1.796565e+07  1.826081e+07  1.774850e+07
6    Redis01         redis-Redis_6379_Redis  (used_memory_peak)     31  5.041321e+07  5.041321e+07  5.041321e+07  5.041321e+07  5.041321e+07
7    Redis01          redis-Redis_6379_Redis  (used_memory_rss)     31  1.203814e+07  1.267302e+07  1.235301e+07  1.257062e+07  1.212416e+07
8    Redis01                      redis-Redis_6379_redis server     31  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00  1.000000e+00
9    Redis01  redis-Redis_6379_Redis  (rdb_changes_since_las...     31  1.203932e+09  1.205473e+09  1.204709e+09  1.205390e+09  1.204013e+09
10   Redis01          redis-Redis_6379_Redis  (redis_git_dirty)     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
11  apache02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
12   Redis01  redis-Redis_6379_Redis  (total_commands_proces...     31  5.494237e+09  5.502697e+09  5.498387e+09  5.502105e+09  5.494686e+09
13   Redis01  redis-Redis_6379_Redis  (total_connections_rec...     31  3.374961e+07  3.376161e+07  3.375566e+07  3.376109e+07  3.375026e+07
14   Redis01        redis-Redis_6379_Redis  (uptime_in_seconds)     31  5.167323e+06  5.169123e+06  5.168223e+06  5.169033e+06  5.167413e+06
15   Redis01          redis-Redis_6379_Redis  (keyspace_misses)     31  6.939000e+03  6.939000e+03  6.939000e+03  6.939000e+03  6.939000e+03
16   Redis01         redis-Redis_6379_Redis  (latest_fork_usec)     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00
17   Redis01                redis-Redis_6379_Redis  (lru_clock)     31  4.919665e+06  4.921465e+06  4.920565e+06  4.921375e+06  4.919755e+06
18   Redis01                redis-Redis_6379_Redis  (maxmemory)     31  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09  4.294967e+09
19   Redis01          redis-Redis_6379_Redis  (pubsub_channels)     31  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00  0.000000e+00```
```

In [ ]:
# In[3]:

# Reuse existing variables: metric_df, metric_agg_all
import pandas as pd

# Allowed root-cause components and time window (inclusive)
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
start_ts = 1615532400
end_ts = 1615534200

# Ensure numeric types
metric_df['timestamp'] = pd.to_numeric(metric_df['timestamp'], errors='coerce').astype('Int64')
metric_df['value'] = pd.to_numeric(metric_df['value'], errors='coerce')

# Filter metric rows to allowed components AND the given time window (inclusive)
metric_window = metric_df[
    metric_df['cmdb_id'].isin(allowed) &
    (metric_df['timestamp'] >= start_ts) &
    (metric_df['timestamp'] <= end_ts)
].copy()

# Join with global aggregates (metric_agg_all) to obtain p95 and p5
# Ensure metric_agg_all exists in kernel; if not, this will raise NameError as expected per rules.
metric_window = metric_window.merge(
    metric_agg_all[['cmdb_id', 'kpi_name', 'p95', 'p5']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomaly: only when p95 and p5 are present. If thresholds are missing, anomaly=False.
metric_window['anomaly'] = False
has_thresh = metric_window['p95'].notna() & metric_window['p5'].notna()
metric_window.loc[has_thresh, 'anomaly'] = (
    (metric_window.loc[has_thresh, 'value'] >= metric_window.loc[has_thresh, 'p95']) |
    (metric_window.loc[has_thresh, 'value'] <= metric_window.loc[has_thresh, 'p5'])
)

# Store the full anomalies table in kernel variable for later steps
metric_window_anoms = metric_window  # full row-level joined table with anomaly flag

# For each (cmdb_id, kpi_name) within the window compute required aggregates
def summarize_group(g):
    total_points = int(len(g))
    anomaly_count = int(g['anomaly'].sum())
    if anomaly_count > 0:
        earliest_anom = int(g.loc[g['anomaly'], 'timestamp'].min())
        latest_anom = int(g.loc[g['anomaly'], 'timestamp'].max())
    else:
        earliest_anom = pd.NA
        latest_anom = pd.NA
    max_value = g['value'].max()
    min_value = g['value'].min()
    # p95/p5 are global thresholds; take first (same within group after merge)
    p95 = g['p95'].iloc[0] if not g['p95'].isna().all() else pd.NA
    p5 = g['p5'].iloc[0] if not g['p5'].isna().all() else pd.NA
    return pd.Series({
        'total_points_in_window': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom,
        'latest_anomaly_timestamp': latest_anom,
        'max_value_in_window': max_value,
        'min_value_in_window': min_value,
        'p95': p95,
        'p5': p5
    })

metric_window_summary = (
    metric_window_anoms
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .apply(summarize_group)
    .reset_index()
)

# Top 20 by anomaly_count descending (compact)
metric_window_summary_top20 = (
    metric_window_summary
    .sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
)

# Display only the compact top-20 aggregate table (metric_window_anoms is stored in kernel)
metric_window_summary_top20

```
Out[3]:
```
Summary of results (using metric.csv, allowed components, window 1615532400–1615534200):

- I stored the full row-level joined results with anomaly flags in the kernel variable metric_window_anoms.

- Top-20 (cmdb_id, kpi_name) by anomaly_count (compact findings):
  - All top-20 rows have total_points_in_window = 31 and anomaly_count = 31 — i.e., every point in the window was marked anomalous for these series.
  - earliest_anomaly_timestamp and latest_anomaly_timestamp (reported as-is) for these top rows are roughly between 1.615532e+09 and 1.615534e+09.
  - p95 and p5 in these groups often equal the observed min/max values, so the series are effectively constant or have thresholds equal to observed values, which caused every point to meet the anomaly condition (value >= p95 or value <= p5).

- Representative top entries (from the top-20):
  1. (IG01, JVM-Memory_7778_JVM_Memory_HeapMemoryMax) — total_points=31, anomalies=31, min=max=p95=p5=2.040136e+09
  2. (IG01, JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed) — total_points=31, anomalies=31, min≈5.784948e+07, max≈5.785396e+07, p95≈5.785396e+07, p5≈5.784948e+07
  3. (IG01, OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT) — total_points=31, anomalies=31, min=max=p95=p5=0.0
  4. (IG01, OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT) — total_points=31, anomalies=31, min=max=p95=p5=0.0
  5. (IG02, JVM-Memory_7778_JVM_Memory_HeapMemoryMax) — total_points=31, anomalies=31, min=max=p95=p5=2.040136e+09
  - Additional top items include MG01/MG02 JVM memory metrics and several Mysql01 network/process metrics, all showing 31/31 anomaly coverage in this window.

Interpretation / note:
- The anomaly rule used global p95/p5 thresholds (computed across the full series) and flagged rows where value >= p95 or <= p5. For many series the p95 and p5 equal the observed values (min==max or near-constant series), therefore every point in the selected window was flagged anomalous. This may indicate constant metrics or threshold computation producing degenerate thresholds; further investigation (e.g., inspect time series in metric_window_anoms) is recommended to determine if these are true anomalies or artifacts of thresholding.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                                           kpi_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp  latest_anomaly_timestamp  max_value_in_window  min_value_in_window           p95            p5
0      IG01           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                    31.0           31.0                1.615532e+09              1.615534e+09         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09
1      IG01        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                    31.0           31.0                1.615532e+09              1.615534e+09         5.785396e+07         5.784948e+07  5.785396e+07  5.784948e+07
2      IG01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
3      IG01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
4      IG02           JVM-Memory_7778_JVM_Memory_HeapMemoryMax                    31.0           31.0                1.615532e+09              1.615534e+09         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09
5      IG02        JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                    31.0           31.0                1.615532e+09              1.615534e+09         5.788538e+07         5.788538e+07  5.788538e+07  5.788538e+07
6      IG02     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
7      IG02       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
8      IG02      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                    31.0           31.0                1.615532e+09              1.615534e+09         1.000000e+00         1.000000e+00  1.000000e+00  1.000000e+00
9      IG02  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                    31.0           31.0                1.615532e+09              1.615534e+09         4.000000e-01         3.000000e-01  4.000000e-01  3.000000e-01
10     IG02         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                    31.0           31.0                1.615532e+09              1.615534e+09         3.000000e+00         3.000000e+00  3.000000e+00  3.000000e+00
11     MG01           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                    31.0           31.0                1.615532e+09              1.615534e+09         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09
12     MG01        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                    31.0           31.0                1.615532e+09              1.615534e+09         5.069495e+08         5.069495e+08  5.069495e+08  5.069495e+08
13     MG02           JVM-Memory_7779_JVM_Memory_HeapMemoryMax                    31.0           31.0                1.615532e+09              1.615534e+09         2.040136e+09         2.040136e+09  2.040136e+09  2.040136e+09
14     MG02        JVM-Memory_7779_JVM_Memory_NoHeapMemoryUsed                    31.0           31.0                1.615532e+09              1.615534e+09         9.471521e+08         9.471521e+08  9.471521e+08  9.471521e+08
15  Mysql01     OSLinux-OSLinux_NETWORK_NETWORK_TCP-CLOSE-WAIT                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
16  Mysql01       OSLinux-OSLinux_NETWORK_NETWORK_TCP-FIN-WAIT                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
17  Mysql01      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                    31.0           31.0                1.615532e+09              1.615534e+09         0.000000e+00         0.000000e+00  0.000000e+00  0.000000e+00
18  Mysql01  OSLinux-OSLinux_PROCESS_zabbix-zabbix_agentd_P...                    31.0           31.0                1.615532e+09              1.615534e+09         4.000000e-01         4.000000e-01  4.000000e-01  4.000000e-01
19  Mysql01         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                    31.0           31.0                1.615532e+09              1.615534e+09         7.000000e+00         7.000000e+00  7.000000e+00  7.000000e+00```
```

In [ ]:
# In[4]:

# Reuse existing kernel variables: metric_window_anoms
import pandas as pd
import pytz

# aggregate per (cmdb_id, kpi_name)
def agg_candidate(g):
    total_points = int(len(g))
    anomaly_count = int(g['anomaly'].sum())
    min_value = g['value'].min()
    max_value = g['value'].max()
    value_range = max_value - min_value if pd.notna(max_value) and pd.notna(min_value) else pd.NA
    if anomaly_count > 0:
        earliest_anom = int(g.loc[g['anomaly'], 'timestamp'].min())
        latest_anom = int(g.loc[g['anomaly'], 'timestamp'].max())
    else:
        earliest_anom = pd.NA
        latest_anom = pd.NA
    # p95 and p5 were joined into metric_window_anoms already (global thresholds)
    p95 = g['p95'].iloc[0] if not g['p95'].isna().all() else pd.NA
    p5 = g['p5'].iloc[0] if not g['p5'].isna().all() else pd.NA
    return pd.Series({
        'total_points_in_window': total_points,
        'anomaly_count': anomaly_count,
        'min_value_in_window': min_value,
        'max_value_in_window': max_value,
        'value_range': value_range,
        'earliest_anomaly_timestamp': earliest_anom,
        'latest_anomaly_timestamp': latest_anom,
        'p95': p95,
        'p5': p5
    })

metric_candidate_ranges = (
    metric_window_anoms
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .apply(agg_candidate)
    .reset_index()
)

# Top 20 by value_range descending (compact)
metric_candidate_top20 = (
    metric_candidate_ranges
    .sort_values('value_range', ascending=False)
    .head(20)
    .reset_index(drop=True)
    # ensure columns order as requested
    [['cmdb_id','kpi_name','total_points_in_window','anomaly_count',
      'min_value_in_window','max_value_in_window','value_range',
      'earliest_anomaly_timestamp','latest_anomaly_timestamp','p95','p5']]
)

# Display only the compact top-20 table; full aggregated table stored in metric_candidate_ranges
metric_candidate_ranges, metric_candidate_top20

```
Out[4]:
```
Summary (plain English):

- I stored the full per-(cmdb_id,kpi_name) aggregated table as metric_candidate_ranges in the kernel.

- The output returned the top 20 candidate metric series sorted by value_range (max - min) within the selected 30‑minute window. Each row includes total_points_in_window, anomaly_count, min/max/value_range, earliest and latest anomaly timestamps, and the global p95/p5 thresholds.

- Key highlights from the top 20 (values shown approximately):
  1. Redis01 OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn — total_points=16, anomaly_count=2, min≈3.39e3, max≈6.205e9, value_range≈6.205e9, p95≈6.2035e9, p5≈4.082e3
  2. Redis01 OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut — total_points=16, anomaly_count=2, min≈3.22e3, max≈5.646e9, value_range≈5.646e9, p95≈5.6447e9, p5≈3.8925e3
  3. IG01 JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — total_points=31, anomaly_count=4, min≈5.35e7, max≈8.73e8, value_range≈8.20e8, p95≈8.71e8, p5≈1.29e8
  4. MG01 JVM-Memory_7779_JVM_Memory_HeapMemoryUsed — total_points=31, anomaly_count=4, min≈2.46e8, max≈1.045e9, value_range≈8.0e8, p95≈9.55e8, p5≈2.61e8
  5. IG02 JVM-Memory_7778_JVM_Memory_HeapMemoryUsed — total_points=31, anomaly_count=4, min≈5.30e7, max≈8.36e8, value_range≈7.83e8, p95≈7.93e8, p5≈8.02e7

- Other notable candidates include MG02 JVM heap usage and several Tomcat JVM memory metrics (value_range on the order of 5e8–7.5e8), and Tomcat request counters (value_range ~1.07e7).

- earliest_anomaly_timestamp and latest_anomaly_timestamp for these top rows fall within the window (numeric timestamps ≈ 1.615532e+09 to 1.615534e+09).

- Interpretation: the top-ranked series by value_range show the largest dynamic change in the window (network packet counters and JVM heap/used memory metrics). anomaly_count varies (some series have few anomalies relative to points; some have many). These top-20 are the strongest candidates for further root-cause investigation.

If you want, I can (a) show the metric_candidate_top20 as exact numeric values in a compact table again, (b) convert earliest/latest anomaly timestamps to readable UTC+8 datetimes, or (c) filter candidates by minimum anomaly_count. Which would you like?

The original code execution output of IPython Kernel is also provided below for reference:

(       cmdb_id                                          kpi_name  total_points_in_window  anomaly_count  min_value_in_window  max_value_in_window   value_range  earliest_anomaly_timestamp  latest_anomaly_timestamp           p95            p5
0         IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryMax                    31.0           31.0         2.040136e+09         2.040136e+09  0.000000e+00                1.615532e+09              1.615534e+09  2.040136e+09  2.040136e+09
1         IG01        JVM-Memory_7778_JVM_Memory_HeapMemoryUsage                    19.0            2.0         5.099400e+00         4.087950e+01  3.578010e+01                1.615534e+09              1.615534e+09  3.665958e+01  8.850060e+00
2         IG01         JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                    31.0            4.0         5.345791e+07         8.732985e+08  8.198406e+08                1.615533e+09              1.615534e+09  8.710115e+08  1.292732e+08
3         IG01       JVM-Memory_7778_JVM_Memory_NoHeapMemoryUsed                    31.0           31.0         5.784948e+07         5.785396e+07  4.480000e+03                1.615532e+09              1.615534e+09  5.785396e+07  5.784948e+07
4         IG01         JVM-Operating System_7778_JVM_JVM_CPULoad                    19.0            2.0         4.445000e-01         6.035000e-01  1.590000e-01                1.615533e+09              1.615533e+09  5.612000e-01  4.466600e-01
...        ...                                               ...                     ...            ...                  ...                  ...           ...                         ...                       ...           ...           ...
1620  apache02                      OSLinux-OSLinux_SWAP_SWAP_Si                    15.0           15.0         0.000000e+00         0.000000e+00  0.000000e+00                1.615532e+09              1.615534e+09  0.000000e+00  0.000000e+00
1621  apache02                      OSLinux-OSLinux_SWAP_SWAP_So                    15.0           15.0         0.000000e+00         0.000000e+00  0.000000e+00                1.615532e+09              1.615534e+09  0.000000e+00  0.000000e+00
1622  apache02  OSLinux-OSLinux_SYSTEM_SYSTEM_Check-DefaultRoute                     3.0            3.0         1.000000e+00         1.000000e+00  0.000000e+00                1.615533e+09              1.615534e+09  1.000000e+00  1.000000e+00
1623  apache02      OSLinux-OSLinux_SYSTEM_SYSTEM_Check-Hostname                     3.0            3.0         1.000000e+00         1.000000e+00  0.000000e+00                1.615533e+09              1.615534e+09  1.000000e+00  1.000000e+00
1624  apache02                OSLinux-OSLinux_ZABBIX_Host_Uptime                    31.0            4.0         6.731074e+06         6.732882e+06  1.808000e+03                1.615532e+09              1.615534e+09  6.732790e+06  6.731164e+06

[1625 rows x 11 columns],      cmdb_id                                           kpi_name  total_points_in_window  anomaly_count  min_value_in_window  max_value_in_window   value_range  earliest_anomaly_timestamp  latest_anomaly_timestamp           p95            p5
0    Redis01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn                    16.0            2.0         3.389000e+03         6.205159e+09  6.205156e+09                1.615533e+09              1.615534e+09  6.203500e+09  4.082000e+03
1    Redis01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut                    16.0            2.0         3.216000e+03         5.646331e+09  5.646327e+09                1.615533e+09              1.615534e+09  5.644747e+09  3.892500e+03
2       IG01          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                    31.0            4.0         5.345791e+07         8.732985e+08  8.198406e+08                1.615533e+09              1.615534e+09  8.710115e+08  1.292732e+08
3       MG01          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                    31.0            4.0         2.456637e+08         1.045402e+09  7.997380e+08                1.615533e+09              1.615534e+09  9.547769e+08  2.609431e+08
4       IG02          JVM-Memory_7778_JVM_Memory_HeapMemoryUsed                    31.0            4.0         5.300347e+07         8.359778e+08  7.829743e+08                1.615533e+09              1.615534e+09  7.930064e+08  8.022794e+07
5       MG02          JVM-Memory_7779_JVM_Memory_HeapMemoryUsed                    31.0            4.0         4.522778e+08         1.214432e+09  7.621537e+08                1.615533e+09              1.615534e+09  1.195943e+09  5.078679e+08
6   Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                     6.0            2.0         1.164584e+09         1.917689e+09  7.531050e+08                1.615532e+09              1.615534e+09  1.859388e+09  1.232884e+09
7   Tomcat04            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                     4.0            2.0         2.269930e+09         3.023035e+09  7.531050e+08                1.615533e+09              1.615534e+09  2.963701e+09  2.304911e+09
8   Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                     6.0            2.0         1.180418e+09         1.905574e+09  7.251560e+08                1.615533e+09              1.615533e+09  1.825952e+09  1.186057e+09
9   Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                     4.0            2.0         2.652695e+09         3.186711e+09  5.340159e+08                1.615533e+09              1.615534e+09  3.184568e+09  2.710570e+09
10  Tomcat02            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                     6.0            2.0         1.000908e+09         1.534924e+09  5.340159e+08                1.615533e+09              1.615534e+09  1.487143e+09  1.004480e+09
11  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMUsedMemory                     6.0            2.0         1.548909e+09         2.018016e+09  4.691068e+08                1.615532e+09              1.615533e+09  2.005648e+09  1.573104e+09
12      IG02        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn                    15.0            2.0         2.160000e+02         4.317608e+08  4.317606e+08                1.615533e+09              1.615534e+09  4.316923e+08  2.244000e+02
13      IG01        OSLinux-OSLinux_NETWORK_ens160_NETPacketsIn                    16.0            2.0         2.300000e+02         4.303565e+08  4.303562e+08                1.615533e+09              1.615534e+09  4.303085e+08  2.337500e+02
14      IG02       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut                    15.0            2.0         2.020000e+02         4.016895e+08  4.016893e+08                1.615533e+09              1.615534e+09  4.016250e+08  2.097000e+02
15      IG01       OSLinux-OSLinux_NETWORK_ens160_NETPacketsOut                    16.0            2.0         2.150000e+02         4.003727e+08  4.003725e+08                1.615533e+09              1.615534e+09  4.003276e+08  2.195000e+02
16  Tomcat01            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                     4.0            2.0         2.169603e+09         2.541933e+09  3.723295e+08                1.615533e+09              1.615534e+09  2.503521e+09  2.180072e+09
17  Tomcat03            Tomcat-MEMORY_7441-MEMORY_JVMFreeMemory                     3.0            2.0         2.748839e+09         3.007202e+09  2.583626e+08                1.615533e+09              1.615534e+09  2.996159e+09  2.763632e+09
18  Tomcat02  Tomcat-Requests_7441-"http-nio-8003"_Processin...                    31.0            4.0         7.987194e+08         8.095891e+08  1.086971e+07                1.615532e+09              1.615534e+09  8.082515e+08  7.992805e+08
19  Tomcat04  Tomcat-Requests_7441-"http-nio-8003"_Processin...                    31.0            4.0         7.617451e+08         7.724532e+08  1.070817e+07                1.615532e+09              1.615534e+09  7.710639e+08  7.623286e+08)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[5]:

# Reuse existing trace_df and allowed list from kernel
import pandas as pd
import pytz

# Ensure allowed list exists; otherwise define (reusing if present)
try:
    allowed  # reuse variable from kernel
except NameError:
    allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
               'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']

# Ensure value is numeric
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Filter to allowed components (global aggregates must be computed on full series of these components)
trace_allowed = trace_df[trace_df['cmdb_id'].isin(allowed)].copy()

# Aggregate global statistics per (cmdb_id, trace_name)
trace_agg_all = (
    trace_allowed
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .agg(
        count='count',
        min='min',
        max='max',
        mean='mean',
        p95=lambda x: x.quantile(0.95) if x.notna().any() else float('nan'),
        p5=lambda x: x.quantile(0.05) if x.notna().any() else float('nan')
    )
    .reset_index()
)

# Top 20 by count descending (compact)
trace_agg_top20 = trace_agg_all.sort_values('count', ascending=False).head(20).reset_index(drop=True)

# Display compact top-20 table
trace_agg_top20

```
Out[5]:
```
Summary (plain English):

- I computed global aggregates across the full trace series for the allowed components and saved the full aggregated table as trace_agg_all in the kernel.

- The returned compact top-20 (sorted by count) shows mostly full-series coverage (count = 31 for many entries; some IG01/IG02 entries have count = 30). Each row includes: count, min, max, mean, p95, p5.

- Key top entries (examples):
  - IG02 / trace.to_Tomcat03.duration_mean — count=31, min≈0.374s, max≈3.275s, mean≈0.746s, p95≈1.159s, p5≈0.433s.
  - IG02 / trace.to_Tomcat03.duration_p95 — count=31, min≈2.022s, max≈5.222s, mean≈2.807s, p95≈4.788s, p5≈2.042s.
  - IG02 / trace.to_Tomcat03.row_count — count=31, min=1, max=311, mean≈242.45, p95=305, p5=192.
  - Tomcat02 / trace.self.duration_mean — count=31, min=0.0s, max≈0.2137s, mean≈0.0987s, p95≈0.1643s, p5≈0.0541s.
  - Tomcat03 / trace.self.row_count — count=31, min=1, max=4360, mean≈3310.8, p95=4103, p5≈2660.5.
  - Tomcat02 / trace.self.row_count — count=31, min=1, max=4215, mean≈3271.4, p95=4150, p5≈2867.5.

- Interpretation: The top traces include inter-service durations (IG02 → Tomcat03 and similar) and high-volume row_count metrics for Tomcat services. Duration p95 values identify high-latency tails (several seconds for IG→Tomcat paths), while row_count metrics show heavy traffic peaks (thousands of rows). These top-20 aggregated metrics are good candidates to investigate for performance/latency root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  count          min          max         mean          p95           p5
0       IG02  trace.to_Tomcat03.duration_mean     31     0.374280     3.275000     0.745819     1.159075     0.432528
1       IG02   trace.to_Tomcat03.duration_p95     31     2.022000     5.221500     2.807153     4.788150     2.041900
2       IG02      trace.to_Tomcat03.row_count     31     1.000000   311.000000   242.451613   305.000000   192.000000
3   Tomcat02         trace.self.duration_mean     31     0.000000     0.213662     0.098685     0.164280     0.054111
4   Tomcat03         trace.self.duration_mean     31     0.059038     1.004000     0.126065     0.178939     0.061365
5   Tomcat03          trace.self.duration_p95     31     0.077850     1.067000     0.812753     1.036775     0.084850
6   Tomcat03        trace.from_IG02.row_count     31     1.000000   311.000000   242.451613   305.000000   192.000000
7   Tomcat03    trace.from_IG02.duration_mean     31     0.374280     3.275000     0.745819     1.159075     0.432528
8   Tomcat03             trace.self.row_count     31     1.000000  4360.000000  3310.806452  4103.000000  2660.500000
9   Tomcat02             trace.self.row_count     31     1.000000  4215.000000  3271.387097  4150.000000  2867.500000
10  Tomcat02          trace.self.duration_p95     31     0.000000     1.097300     0.900511     1.047500     0.093050
11  Tomcat03     trace.from_IG02.duration_p95     31     2.022000     5.221500     2.807153     4.788150     2.041900
12      IG01   trace.to_Tomcat01.duration_p95     30     1.429700     7.055750     2.838473     4.520330     2.056660
13      IG01   trace.to_Tomcat02.duration_p95     30     2.036400     5.625100     2.894972     5.361768     2.065433
14      IG01             trace.self.row_count     30  1686.000000  2493.000000  2017.166667  2394.750000  1733.700000
15      IG01         trace.self.duration_mean     30     0.385405     1.441085     0.678104     1.122560     0.447633
16      IG02  trace.to_Tomcat02.duration_mean     30     0.307280     1.384904     0.686262     1.084422     0.418688
17      IG02      trace.to_Tomcat01.row_count     30   176.000000   341.000000   248.366667   315.650000   203.150000
18      IG02      trace.to_Tomcat02.row_count     30   207.000000   315.000000   251.200000   312.650000   209.700000
19      IG01      trace.to_Tomcat04.row_count     30   194.000000   323.000000   250.666667   302.300000   204.950000```
```

In [ ]:
# In[6]:

# Reuse existing trace_df and trace_agg_all from kernel
import pandas as pd

# Allowed components and time window (inclusive)
allowed = ['apache01','apache02','Tomcat01','Tomcat02','Tomcat03','Tomcat04',
           'MG01','MG02','IG01','IG02','Mysql01','Mysql02','Redis01','Redis02']
start_ts = 1615532400
end_ts = 1615534200

# Ensure numeric types
trace_df['timestamp'] = pd.to_numeric(trace_df['timestamp'], errors='coerce').astype('Int64')
trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')

# Filter rows to allowed components and the specified time window
trace_window = trace_df[
    trace_df['cmdb_id'].isin(allowed) &
    (trace_df['timestamp'] >= start_ts) &
    (trace_df['timestamp'] <= end_ts)
].copy()

# Join with global aggregates to get p95 and p5 (trace_agg_all expected in kernel)
trace_window = trace_window.merge(
    trace_agg_all[['cmdb_id', 'trace_name', 'p95', 'p5']],
    on=['cmdb_id', 'trace_name'],
    how='left'
)

# Mark anomaly where thresholds exist: value >= p95 OR value <= p5
trace_window['anomaly'] = False
has_thresh = trace_window['p95'].notna() & trace_window['p5'].notna()
trace_window.loc[has_thresh, 'anomaly'] = (
    (trace_window.loc[has_thresh, 'value'] >= trace_window.loc[has_thresh, 'p95']) |
    (trace_window.loc[has_thresh, 'value'] <= trace_window.loc[has_thresh, 'p5'])
)

# Aggregate per (cmdb_id, trace_name)
def agg_trace_group(g):
    total_points = int(len(g))
    anomaly_count = int(g['anomaly'].sum())
    min_val = g['value'].min()
    max_val = g['value'].max()
    value_range = max_val - min_val if pd.notna(min_val) and pd.notna(max_val) else pd.NA
    if anomaly_count > 0:
        earliest_anom = int(g.loc[g['anomaly'], 'timestamp'].min())
        latest_anom = int(g.loc[g['anomaly'], 'timestamp'].max())
    else:
        earliest_anom = pd.NA
        latest_anom = pd.NA
    p95 = g['p95'].iloc[0] if not g['p95'].isna().all() else pd.NA
    p5 = g['p5'].iloc[0] if not g['p5'].isna().all() else pd.NA
    return pd.Series({
        'total_points_in_window': total_points,
        'anomaly_count': anomaly_count,
        'earliest_anomaly_timestamp': earliest_anom,
        'latest_anomaly_timestamp': latest_anom,
        'min_value_in_window': min_val,
        'max_value_in_window': max_val,
        'value_range': value_range,
        'p95': p95,
        'p5': p5
    })

trace_window_anoms = (
    trace_window
    .groupby(['cmdb_id', 'trace_name'], dropna=False)
    .apply(agg_trace_group)
    .reset_index()
)

# Prepare compact top-20 by anomaly_count descending
trace_window_top20 = (
    trace_window_anoms
    .sort_values(['anomaly_count', 'total_points_in_window'], ascending=[False, False])
    .head(20)
    .reset_index(drop=True)
    [['cmdb_id','trace_name','total_points_in_window','anomaly_count',
      'earliest_anomaly_timestamp','latest_anomaly_timestamp',
      'min_value_in_window','max_value_in_window','value_range','p95','p5']]
)

# Display only the compact top-20 table; full aggregated table is stored as trace_window_anoms
trace_window_top20

```
Out[6]:
```
Summary (plain English):

- I stored the full per-row joined and labeled results as trace_window_anoms in the kernel.

- Top 20 trace series by anomaly_count (compact findings):
  - The highest anomaly_count = 5 occurs for:
    - MG01 / trace.to_dockerB2.row_count — 30 points in window, 5 anomalies, min=110, max=296, range=186, p95=268.0, p5=132.85 (earliest/latest anomaly timestamps ≈ 1.615533e+09).
    - MG02 / trace.to_dockerA1.row_count — 30 points, 5 anomalies, min=118, max=288, range=170, p95=235.65, p5=124.0.
    - MG02 / trace.to_dockerB1.row_count — 30 points, 5 anomalies, min=148, max=328, range=180, p95=254.3, p5=151.0.
    - IG01 / trace.from_root.row_count — 10 points, 5 anomalies, min=1, max=10, range=9, p95=8.65, p5=1.0.
  - Several IG02 → Tomcat03 series have anomaly_count = 4:
    - IG02 / trace.to_Tomcat03.duration_mean — 31 pts, 4 anomalies, min≈0.374s, max≈3.275s, range≈2.9007s, p95≈1.159, p5≈0.4325.
    - IG02 / trace.to_Tomcat03.duration_p95 and trace.to_Tomcat03.row_count — 31 pts, 4 anomalies, large ranges for duration_p95 and row_count.
  - Tomcat services also appear with anomaly_count = 4:
    - Tomcat02 / trace.self.duration_mean, trace.self.duration_p95, trace.self.row_count — 31 pts each, 4 anomalies, large max values for row_count (up to ~4215) and significant duration p95 values.
    - Tomcat03 / trace.from_IG02.* and trace.self.* — similar patterns (31 pts, 4 anomalies).
  - IG01 entries (self.duration_mean, self.duration_p95, self.row_count, to_Tomcat01.duration_mean) show anomaly_count = 4 with 30 points in window.

- Timestamps:
  - Earliest and latest anomaly timestamps for these top series fall within the analysis window (numeric timestamps ≈ 1.615533e+09 to 1.615534e+09).

- Interpretation:
  - Anomalies were flagged using global p95/p5 thresholds (computed across full series). The top candidates include high row_count spikes on MG and Tomcat services and high-duration tails for IG→Tomcat paths.
  - MG→docker and IG→Tomcat interactions, plus Tomcat self row_count/duration metrics, are the strongest trace-level anomaly candidates in the 30‑minute window.

- Suggested next steps:
  - Inspect raw trace rows for the top candidates in trace_window_anoms (timestamps and surrounding context).
  - Cross-correlate these trace anomalies with metric and log anomalies (kernel variables metric_window_anoms and metric_candidate_ranges) to narrow root cause.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id                       trace_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp  latest_anomaly_timestamp  min_value_in_window  max_value_in_window  value_range          p95           p5
0       MG01      trace.to_dockerB2.row_count                    30.0            5.0                1.615533e+09              1.615533e+09           110.000000           296.000000   186.000000   268.000000   132.850000
1       MG02      trace.to_dockerA1.row_count                    30.0            5.0                1.615533e+09              1.615534e+09           118.000000           288.000000   170.000000   235.650000   124.000000
2       MG02      trace.to_dockerB1.row_count                    30.0            5.0                1.615533e+09              1.615533e+09           148.000000           328.000000   180.000000   254.300000   151.000000
3       IG01        trace.from_root.row_count                    10.0            5.0                1.615532e+09              1.615534e+09             1.000000            10.000000     9.000000     8.650000     1.000000
4       IG02  trace.to_Tomcat03.duration_mean                    31.0            4.0                1.615533e+09              1.615534e+09             0.374280             3.275000     2.900720     1.159075     0.432528
5       IG02   trace.to_Tomcat03.duration_p95                    31.0            4.0                1.615533e+09              1.615534e+09             2.022000             5.221500     3.199500     4.788150     2.041900
6       IG02      trace.to_Tomcat03.row_count                    31.0            4.0                1.615533e+09              1.615534e+09             1.000000           311.000000   310.000000   305.000000   192.000000
7   Tomcat02         trace.self.duration_mean                    31.0            4.0                1.615533e+09              1.615534e+09             0.000000             0.213662     0.213662     0.164280     0.054111
8   Tomcat02          trace.self.duration_p95                    31.0            4.0                1.615533e+09              1.615534e+09             0.000000             1.097300     1.097300     1.047500     0.093050
9   Tomcat02             trace.self.row_count                    31.0            4.0                1.615533e+09              1.615534e+09             1.000000          4215.000000  4214.000000  4150.000000  2867.500000
10  Tomcat03    trace.from_IG02.duration_mean                    31.0            4.0                1.615533e+09              1.615534e+09             0.374280             3.275000     2.900720     1.159075     0.432528
11  Tomcat03     trace.from_IG02.duration_p95                    31.0            4.0                1.615533e+09              1.615534e+09             2.022000             5.221500     3.199500     4.788150     2.041900
12  Tomcat03        trace.from_IG02.row_count                    31.0            4.0                1.615533e+09              1.615534e+09             1.000000           311.000000   310.000000   305.000000   192.000000
13  Tomcat03         trace.self.duration_mean                    31.0            4.0                1.615533e+09              1.615534e+09             0.059038             1.004000     0.944962     0.178939     0.061365
14  Tomcat03          trace.self.duration_p95                    31.0            4.0                1.615533e+09              1.615534e+09             0.077850             1.067000     0.989150     1.036775     0.084850
15  Tomcat03             trace.self.row_count                    31.0            4.0                1.615533e+09              1.615534e+09             1.000000          4360.000000  4359.000000  4103.000000  2660.500000
16      IG01         trace.self.duration_mean                    30.0            4.0                1.615533e+09              1.615534e+09             0.385405             1.441085     1.055680     1.122560     0.447633
17      IG01          trace.self.duration_p95                    30.0            4.0                1.615533e+09              1.615534e+09             1.593500             5.807000     4.213500     4.915715     2.067900
18      IG01             trace.self.row_count                    30.0            4.0                1.615533e+09              1.615534e+09          1686.000000          2493.000000   807.000000  2394.750000  1733.700000
19      IG01  trace.to_Tomcat01.duration_mean                    30.0            4.0                1.615533e+09              1.615534e+09             0.391945             1.641587     1.249642     0.961304     0.412264```
```

In [ ]:
# In[7]:

# Reuse kernel variables: metric_window_anoms, trace_window_anoms
import pandas as pd
import numpy as np

# 1) From metric_window_anoms: per-component metric-series anomaly counts and earliest anomaly ts
# Determine which (cmdb_id, kpi_name) had any anomaly in the window
metric_series_any_anom = (
    metric_window_anoms
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['anomaly']
    .any()
    .reset_index(name='series_has_anom')
)

# Count distinct kpi_name with anomaly per component
metric_anom_counts = (
    metric_series_any_anom[metric_series_any_anom['series_has_anom']]
    .groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='num_metric_series_with_anomaly')
)

# Earliest anomaly timestamp per component from row-level anomalies
metric_earliest_ts = (
    metric_window_anoms[metric_window_anoms['anomaly']]
    .groupby('cmdb_id', dropna=False)['timestamp']
    .min()
    .reset_index(name='earliest_metric_anom_ts')
)

# 2) From trace_window_anoms: per-component trace-series anomaly counts and earliest anomaly ts
trace_anom_counts = (
    trace_window_anoms[trace_window_anoms['anomaly_count'] > 0]
    .groupby('cmdb_id', dropna=False)
    .size()
    .reset_index(name='num_trace_series_with_anomaly')
)

# earliest_anomaly_timestamp already exists per series; take min per component and rename
trace_earliest_ts = (
    trace_window_anoms[trace_window_anoms['earliest_anomaly_timestamp'].notna()]
    .groupby('cmdb_id', dropna=False)['earliest_anomaly_timestamp']
    .min()
    .reset_index()
    .rename(columns={'earliest_anomaly_timestamp': 'earliest_trace_anom_ts'})
)

# 3) Build union of components present in either source
all_cmdbs = pd.Series(
    pd.unique(
        pd.concat([
            metric_window_anoms['cmdb_id'].dropna().astype(str),
            trace_window_anoms['cmdb_id'].dropna().astype(str)
        ])
    )
).reset_index(drop=True)

component_anom_summary = pd.DataFrame({'cmdb_id': all_cmdbs}).drop_duplicates().reset_index(drop=True)

# Merge counts and earliest timestamps
component_anom_summary = component_anom_summary.merge(metric_anom_counts, on='cmdb_id', how='left')
component_anom_summary = component_anom_summary.merge(metric_earliest_ts, on='cmdb_id', how='left')
component_anom_summary = component_anom_summary.merge(trace_anom_counts, on='cmdb_id', how='left')
component_anom_summary = component_anom_summary.merge(trace_earliest_ts, on='cmdb_id', how='left')

# Fill missing counts as 0 and keep earliest timestamps as NULL (NaN)
component_anom_summary['num_metric_series_with_anomaly'] = component_anom_summary['num_metric_series_with_anomaly'].fillna(0).astype(int)
component_anom_summary['num_trace_series_with_anomaly'] = component_anom_summary['num_trace_series_with_anomaly'].fillna(0).astype(int)

# 4) Store full per-component summary in kernel variable (already assigned)
# Compute earliest of the two timestamps for sorting; keep NaN if both missing
component_anom_summary['earliest_combined_anom_ts'] = component_anom_summary[['earliest_metric_anom_ts','earliest_trace_anom_ts']].min(axis=1, skipna=True)

# Sort by earliest_combined_anom_ts ascending, placing NaN at the end
component_anom_summary_sorted = component_anom_summary.sort_values(by='earliest_combined_anom_ts', na_position='last').reset_index(drop=True)

# Prepare top 10 output with required columns
component_anom_top10 = component_anom_summary_sorted[['cmdb_id',
                                                     'num_metric_series_with_anomaly',
                                                     'earliest_metric_anom_ts',
                                                     'num_trace_series_with_anomaly',
                                                     'earliest_trace_anom_ts']].head(10)

# Display only the compact top-10 table; full aggregated table stored in component_anom_summary
component_anom_top10

```
Out[7]:
```
Summary (plain English):

- I stored the full per-component table as component_anom_summary in the kernel.

Top 10 components (sorted by earliest observed anomaly), showing: cmdb_id, num_metric_series_with_anomaly, earliest_metric_anom_ts, num_trace_series_with_anomaly, earliest_trace_anom_ts:

- IG01 — 88 metric series, earliest_metric_anom_ts = 1615532400; 18 trace series, earliest_trace_anom_ts = 1.615532e+09  
- IG02 — 88 metric series, earliest_metric_anom_ts = 1615532400; 18 trace series, earliest_trace_anom_ts = 1.615533e+09  
- MG01 — 92 metric series, earliest_metric_anom_ts = 1615532400; 42 trace series, earliest_trace_anom_ts = 1.615532e+09  
- MG02 — 93 metric series, earliest_metric_anom_ts = 1615532400; 42 trace series, earliest_trace_anom_ts = 1.615532e+09  
- Mysql01 — 224 metric series, earliest_metric_anom_ts = 1615532400; 0 trace series, earliest_trace_anom_ts = NULL  
- Mysql02 — 224 metric series, earliest_metric_anom_ts = 1615532400; 0 trace series, earliest_trace_anom_ts = NULL  
- Redis01 — 113 metric series, earliest_metric_anom_ts = 1615532400; 0 trace series, earliest_trace_anom_ts = NULL  
- Redis02 — 110 metric series, earliest_metric_anom_ts = 1615532400; 0 trace series, earliest_trace_anom_ts = NULL  
- Tomcat01 — 105 metric series, earliest_metric_anom_ts = 1615532400; 18 trace series, earliest_trace_anom_ts = 1.615532e+09  
- Tomcat02 — 108 metric series, earliest_metric_anom_ts = 1615532400; 18 trace series, earliest_trace_anom_ts = 1.615533e+09  

Interpretation / next step:
- Components with both metric and trace anomalies (priority candidates): IG01, IG02, MG01, MG02, Tomcat01, Tomcat02 — investigate these first.  
- Some components (Mysql01/02, Redis01/02) show many metric-series anomalies but no trace-series anomalies in the trace data; follow up using metric_window_anoms and metric_candidate_ranges for those.  
- Earliest metric anomalies are at 1615532400 (the window start) for all listed components; trace earliest anomaly timestamps are around 1.615532e+09–1.615533e+09.

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id  num_metric_series_with_anomaly  earliest_metric_anom_ts  num_trace_series_with_anomaly  earliest_trace_anom_ts
0      IG01                              88               1615532400                             18            1.615532e+09
1      IG02                              88               1615532400                             18            1.615533e+09
2      MG01                              92               1615532400                             42            1.615532e+09
3      MG02                              93               1615532400                             42            1.615532e+09
4   Mysql01                             224               1615532400                              0                     NaN
5   Mysql02                             224               1615532400                              0                     NaN
6   Redis01                             113               1615532400                              0                     NaN
7   Redis02                             110               1615532400                              0                     NaN
8  Tomcat01                             105               1615532400                             18            1.615532e+09
9  Tomcat02                             108               1615532400                             18            1.615533e+09```
```

In [ ]:
# In[8]:

# Reuse kernel variables: metric_candidate_ranges, trace_window_anoms, log_df
import pandas as pd

# 1) Metric details for Tomcat03
tomcat03_metric_details = metric_candidate_ranges[
    (metric_candidate_ranges['cmdb_id'] == 'Tomcat03') &
    (metric_candidate_ranges['anomaly_count'] > 0)
].copy()

# Sort as requested and prepare compact top-20 for display
tomcat03_metric_details_top20 = tomcat03_metric_details.sort_values(
    by=['anomaly_count', 'value_range'], ascending=[False, False]
).head(20).reset_index(drop=True)[[
    'cmdb_id', 'kpi_name', 'total_points_in_window', 'anomaly_count',
    'min_value_in_window', 'max_value_in_window', 'value_range',
    'earliest_anomaly_timestamp', 'latest_anomaly_timestamp', 'p95', 'p5'
]]

# 2) Trace details for Tomcat03
tomcat03_trace_details = trace_window_anoms[
    (trace_window_anoms['cmdb_id'] == 'Tomcat03') &
    (trace_window_anoms['anomaly_count'] > 0)
].copy()

# Sort by earliest_anomaly_timestamp ascending and prepare compact top-20
tomcat03_trace_details_top20 = tomcat03_trace_details.sort_values(
    by=['earliest_anomaly_timestamp'], ascending=True
).head(20).reset_index(drop=True)[[
    'cmdb_id', 'trace_name', 'total_points_in_window', 'anomaly_count',
    'earliest_anomaly_timestamp', 'latest_anomaly_timestamp',
    'min_value_in_window', 'max_value_in_window', 'value_range', 'p95', 'p5'
]]

# 3) Log summary for Tomcat03 within the window
start_ts = 1615532400
end_ts = 1615534200

# Ensure numeric and filter
log_df['timestamp'] = pd.to_numeric(log_df['timestamp'], errors='coerce').astype('Int64')
log_df['value'] = pd.to_numeric(log_df['value'], errors='coerce')

tomcat03_logs_window = log_df[
    (log_df['cmdb_id'] == 'Tomcat03') &
    (log_df['timestamp'] >= start_ts) &
    (log_df['timestamp'] <= end_ts)
].copy()

# Aggregate by log_name
tomcat03_log_summary = (
    tomcat03_logs_window
    .groupby('log_name', dropna=False)
    .agg(
        count_rows=('value', 'count'),
        min_value=('value', 'min'),
        max_value=('value', 'max'),
        mean_value=('value', 'mean'),
        earliest_timestamp=('timestamp', 'min'),
        latest_timestamp=('timestamp', 'max')
    )
    .reset_index()
)

# Sort by count_rows desc and limit to top 20 for display
tomcat03_log_summary_top20 = tomcat03_log_summary.sort_values('count_rows', ascending=False).head(20).reset_index(drop=True)

# Return the three compact tables only (top-20 views for metric and trace; top-20 for log summary)
tomcat03_metric_details_top20, tomcat03_trace_details_top20, tomcat03_log_summary_top20

```
Out[8]:
```
Summary for Tomcat03 (compact results stored in kernel variables):

Metric highlights (tomcat03_metric_details):
- Many Tomcat03 KPI series were flagged anomalous across the window; several series have anomaly_count = 31 with min == max == p95 == p5 (i.e., constant values caused every point to meet the anomaly rule). Examples:
  - Tomcat-Requests..."ErrorCount": min=max=339 (31/31 anomalies).
  - Tomcat-Requests..."MaxTimeRe...": min=max=364719 (31/31).
  - Multiple Tomcat-Sessions metrics (ActiveCounter, KeepaliveCounter, RejectedSessions, etc.): all 0 with 31/31 anomalies.
  - Thread metrics: CurrentThreads=107 (31/31), MaxThreads=500 (31/31).
  - OSLinux memory/process metrics: MEMUsedPerc=98 (21/21), PROCNoZombies=1 (21/21), PROCPPMem=58 (21/21).
- Interpretation: several metrics are effectively constant in the window (or thresholds equal observed values), producing full-window anomaly flags. Other metrics with non-zero ranges are also present but lower in ranking.

Trace highlights (tomcat03_trace_details):
- Multiple trace series show anomalies (mostly anomaly_count = 4) and significant variation:
  - trace.self.row_count — 31 points, anomaly_count=4, min=1, max=4360, range=4359, p95=4103, p5≈2660.5 (very large traffic peaks).
  - trace.from_IG02.row_count and trace.from_IG01.row_count — 31/30 points, anomaly_count=4, high row_count peaks (max ≈ 311 and up).
  - Several duration metrics show high tails (p95 values multiple seconds): e.g., trace.from_IG02.duration_p95, trace.to_MG02.duration_p95, trace.from_IG01.duration_p95 with large p95 and wide ranges.
- Interpretation: trace data indicates traffic spikes (row_count) and high-latency tails on IG→Tomcat and Tomcat→MG paths — strong trace-level candidates for performance root cause.

Log summary (tomcat03_log_summary):
- log.error_count: 31 rows, all values = 0 (no logged errors in this window).
- log.row_count: 31 rows, min=7, max=620, mean≈491.6 (substantial per-minute row counts; matches trace row_count spikes).
- log.gc_pause_max: 25 rows, min≈0.02, max≈0.11, mean≈0.063 s (some GC pauses observed, not huge).

Overall interpretation and next steps:
- Trace anomalies (high row_count and high p95 durations) point to traffic/latency issues involving Tomcat03 (especially inbound from IG01/IG02 and interactions with MG nodes).
- Many metric anomalies are degenerate (constant series flagged because global p95/p5 equal observed values); these require verifying threshold logic or inspecting raw time series to avoid false positives.
- Recommended next steps: inspect raw rows for the top trace candidates and the corresponding metric series in metric_window_anoms and the stored tomcat03_* detail tables to correlate timestamps and determine causality.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id                                           kpi_name  total_points_in_window  anomaly_count  min_value_in_window  max_value_in_window  value_range  earliest_anomaly_timestamp  latest_anomaly_timestamp       p95        p5
0   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...                    31.0           31.0                339.0                339.0          0.0                1.615532e+09              1.615534e+09     339.0     339.0
1   Tomcat03  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...                    31.0           31.0             364719.0             364719.0          0.0                1.615532e+09              1.615534e+09  364719.0  364719.0
2   Tomcat03    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
3   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
4   Tomcat03  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
5   Tomcat03        Tomcat-Sessions_7441--_SESSIONActiveCounter                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
6   Tomcat03     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
7   Tomcat03     Tomcat-Sessions_7441--_SESSIONRejectedSessions                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
8   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
9   Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
10  Tomcat03  Tomcat-Sessions_7441--log.Home_IS_UNDEFINED_SE...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
11  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
12  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
13  Tomcat03  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...                    31.0           31.0                  0.0                  0.0          0.0                1.615532e+09              1.615534e+09       0.0       0.0
14  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...                    31.0           31.0                107.0                107.0          0.0                1.615532e+09              1.615534e+09     107.0     107.0
15  Tomcat03  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...                    31.0           31.0                500.0                500.0          0.0                1.615532e+09              1.615534e+09     500.0     500.0
16  Tomcat03       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc                    21.0           21.0                 98.0                 98.0          0.0                1.615532e+09              1.615534e+09      98.0      98.0
17  Tomcat03      OSLinux-OSLinux_PROCESS_PROCESS_PROCNoZombies                    21.0           21.0                  1.0                  1.0          0.0                1.615532e+09              1.615534e+09       1.0       1.0
18  Tomcat03          OSLinux-OSLinux_PROCESS_PROCESS_PROCPPMem                    21.0           21.0                 58.0                 58.0          0.0                1.615532e+09              1.615534e+09      58.0      58.0
19  Tomcat03         OSLinux-OSLinux_PROCESS_zabbix_PROCPPCount                    21.0           21.0                  3.0                  3.0          0.0                1.615532e+09              1.615534e+09       3.0       3.0,      cmdb_id                     trace_name  total_points_in_window  anomaly_count  earliest_anomaly_timestamp  latest_anomaly_timestamp  min_value_in_window  max_value_in_window  value_range          p95           p5
0   Tomcat03      trace.from_IG01.row_count                    30.0            4.0                1.615532e+09              1.615534e+09           207.000000           322.000000   115.000000   311.650000   212.800000
1   Tomcat03      trace.from_root.row_count                     7.0            2.0                1.615532e+09              1.615533e+09             2.000000            34.000000    32.000000    31.900000     3.200000
2   Tomcat03  trace.from_root.duration_mean                     7.0            2.0                1.615533e+09              1.615533e+09             0.004188             0.649000     0.644812     0.564078     0.032707
3   Tomcat03   trace.from_root.duration_p95                     7.0            2.0                1.615533e+09              1.615534e+09             0.024000             2.121900     2.097900     1.796400     0.136080
4   Tomcat03           trace.self.row_count                    31.0            4.0                1.615533e+09              1.615534e+09             1.000000          4360.000000  4359.000000  4103.000000  2660.500000
5   Tomcat03      trace.from_IG02.row_count                    31.0            4.0                1.615533e+09              1.615534e+09             1.000000           311.000000   310.000000   305.000000   192.000000
6   Tomcat03        trace.to_MG02.row_count                    30.0            4.0                1.615533e+09              1.615533e+09           191.000000           325.000000   134.000000   306.350000   201.050000
7   Tomcat03  trace.from_IG01.duration_mean                    30.0            4.0                1.615533e+09              1.615534e+09             0.362004             1.262055     0.900051     1.126677     0.394847
8   Tomcat03    trace.to_MG02.duration_mean                    30.0            4.0                1.615533e+09              1.615534e+09             0.209179             1.158715     0.949536     1.001974     0.279971
9   Tomcat03   trace.from_IG02.duration_p95                    31.0            4.0                1.615533e+09              1.615534e+09             2.022000             5.221500     3.199500     4.788150     2.041900
10  Tomcat03     trace.to_MG01.duration_p95                    30.0            4.0                1.615533e+09              1.615534e+09             1.170800             5.213050     4.042250     4.514155     1.183917
11  Tomcat03    trace.to_MG01.duration_mean                    30.0            4.0                1.615533e+09              1.615534e+09             0.293603             1.183526     0.889923     1.027802     0.307522
12  Tomcat03     trace.to_MG02.duration_p95                    30.0            4.0                1.615533e+09              1.615534e+09             1.065750             5.430600     4.364850     4.424917     1.127520
13  Tomcat03   trace.from_IG01.duration_p95                    30.0            4.0                1.615533e+09              1.615534e+09             1.267500             5.589300     4.321800     4.408053     1.621645
14  Tomcat03  trace.from_IG02.duration_mean                    31.0            4.0                1.615533e+09              1.615534e+09             0.374280             3.275000     2.900720     1.159075     0.432528
15  Tomcat03       trace.self.duration_mean                    31.0            4.0                1.615533e+09              1.615534e+09             0.059038             1.004000     0.944962     0.178939     0.061365
16  Tomcat03        trace.to_MG01.row_count                    30.0            4.0                1.615533e+09              1.615534e+09           160.000000           320.000000   160.000000   309.300000   190.450000
17  Tomcat03        trace.self.duration_p95                    31.0            4.0                1.615533e+09              1.615534e+09             0.077850             1.067000     0.989150     1.036775     0.084850,            log_name  count_rows  min_value  max_value  mean_value  earliest_timestamp  latest_timestamp
0   log.error_count          31       0.00       0.00    0.000000          1615532400        1615534200
1     log.row_count          31       7.00     620.00  491.612903          1615532400        1615534200
2  log.gc_pause_max          25       0.02       0.11    0.063200          1615532460        1615534140)```
```

In [ ]:
# In[9]:

# Reuse kernel variables where possible; fall back to reconstructing per-row trace join if needed.
import pandas as pd

start_ts = 1615532400
end_ts = 1615534200
cmdb_filter = ['IG02', 'Tomcat03']

# Preferred variable with per-row joined trace data may be 'trace_window' (per-row) or else reconstruct from trace_df + trace_agg_all
try:
    # Use existing per-row joined table if available
    trace_rows = trace_window  # expected to be the per-row joined DataFrame used earlier
    _ = trace_rows.shape  # trigger NameError if not defined
except NameError:
    # Reconstruct per-row joined trace data from raw trace_df and global aggregates trace_agg_all
    # Ensure trace_df and trace_agg_all exist in kernel
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
    trace_rows = trace_df.merge(
        trace_agg_all[['cmdb_id', 'trace_name', 'p95', 'p5']],
        on=['cmdb_id', 'trace_name'],
        how='left'
    )
    # compute anomaly flag if thresholds present
    trace_rows['anomaly'] = False
    has_thresh = trace_rows['p95'].notna() & trace_rows['p5'].notna()
    trace_rows.loc[has_thresh, 'anomaly'] = (
        (trace_rows.loc[has_thresh, 'value'] >= trace_rows.loc[has_thresh, 'p95']) |
        (trace_rows.loc[has_thresh, 'value'] <= trace_rows.loc[has_thresh, 'p5'])
    )

# Ensure timestamp and value are numeric
trace_rows['timestamp'] = pd.to_numeric(trace_rows['timestamp'], errors='coerce').astype('Int64')
trace_rows['value'] = pd.to_numeric(trace_rows['value'], errors='coerce')

# Filter by cmdb_id, time window, and trace_name patterns
pattern_mask = (
    trace_rows['trace_name'].str.contains('to_Tomcat03', na=False) |
    trace_rows['trace_name'].str.contains('from_IG02', na=False) |
    trace_rows['trace_name'].str.startswith('trace.self', na=False)
)

filtered = trace_rows[
    trace_rows['cmdb_id'].isin(cmdb_filter) &
    (trace_rows['timestamp'] >= start_ts) &
    (trace_rows['timestamp'] <= end_ts) &
    pattern_mask
].copy()

# Determine anomaly_flag and anomaly_type per row (respecting existing 'anomaly' if present)
# Recompute explicitly to ensure correctness
has_thresh = filtered['p95'].notna() & filtered['p5'].notna()
filtered['anomaly_flag'] = False
filtered.loc[has_thresh, 'anomaly_flag'] = (
    (filtered.loc[has_thresh, 'value'] >= filtered.loc[has_thresh, 'p95']) |
    (filtered.loc[has_thresh, 'value'] <= filtered.loc[has_thresh, 'p5'])
)

def anom_type(row):
    if pd.isna(row['p95']) or pd.isna(row['p5']) or pd.isna(row['value']):
        return pd.NA
    if row['value'] >= row['p95']:
        return 'high'
    if row['value'] <= row['p5']:
        return 'low'
    return pd.NA

filtered['anomaly_type'] = filtered.apply(anom_type, axis=1)

# Select requested columns and sort by timestamp ascending, limit to top 20
tomcat03_ig02_trace_rows = filtered[
    ['timestamp', 'cmdb_id', 'trace_name', 'value', 'p95', 'p5', 'anomaly_flag', 'anomaly_type']
].sort_values('timestamp', ascending=True).head(20).reset_index(drop=True)

# Return the result (stored in kernel variable tomcat03_ig02_trace_rows)
tomcat03_ig02_trace_rows

```
Out[9]:
```
Summary of retrieved trace rows (stored in kernel variable 'tomcat03_ig02_trace_rows'):

- Scope: rows for cmdb_id in ['IG02','Tomcat03'] with trace_name matching contains "to_Tomcat03", contains "from_IG02", or starts with "trace.self", restricted to timestamps 1615532400–1615534200.

- Returned rows: 20 rows, sorted by timestamp ascending. They include two timestamp groups shown here: 1615532400 and 1615532460 (the earliest rows in the incident window).

- Trace names present: trace.self.duration_mean, trace.self.duration_p95, trace.self.row_count, trace.to_Tomcat03.duration_mean, trace.to_Tomcat03.duration_p95, trace.to_Tomcat03.row_count, trace.from_IG02.duration_mean, trace.from_IG02.duration_p95, trace.from_IG02.row_count, and trace.from_root variations for Tomcat03.

- Anomaly results: none of the 20 returned rows are marked anomalous (anomaly_flag = False for all; anomaly_type is NA). In other words, for these specific timestamps, every value is between the global p5 and p95 thresholds (value < p95 and value > p5).

- Representative examples:
  - IG02 trace.self.row_count at 1615532400: value=1907, p95=2435.15, p5=1703.30 → not anomalous.
  - IG02 trace.to_Tomcat03.row_count at 1615532400: value=205, p95=305, p5=192 → not anomalous.
  - Tomcat03 trace.self.row_count at 1615532400: value=2756, p95=4103, p5=2660.5 → not anomalous.
  - Durations (e.g., trace.to_Tomcat03.duration_mean ≈ 0.72–0.79s) are below their p95 thresholds.

- Interpretation: although earlier aggregated analysis showed some trace-series with anomalies in the window, these particular sampled per-minute rows at the two earliest timestamps are within the global thresholds. Use the stored variable 'tomcat03_ig02_trace_rows' to inspect additional timestamps, or correlate with metric/log anomalies for deeper analysis.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                       trace_name        value          p95           p5  anomaly_flag anomaly_type
0   1615532400      IG02         trace.self.duration_mean     0.809432     1.028284     0.414144         False         <NA>
1   1615532400      IG02          trace.self.duration_p95     3.264300     4.404725     2.052650         False         <NA>
2   1615532400      IG02             trace.self.row_count  1907.000000  2435.150000  1703.300000         False         <NA>
3   1615532400      IG02  trace.to_Tomcat03.duration_mean     0.724941     1.159075     0.432528         False         <NA>
4   1615532400      IG02   trace.to_Tomcat03.duration_p95     3.097800     4.788150     2.041900         False         <NA>
5   1615532400      IG02      trace.to_Tomcat03.row_count   205.000000   305.000000   192.000000         False         <NA>
6   1615532400  Tomcat03    trace.from_IG02.duration_mean     0.724941     1.159075     0.432528         False         <NA>
7   1615532400  Tomcat03     trace.from_IG02.duration_p95     3.097800     4.788150     2.041900         False         <NA>
8   1615532400  Tomcat03        trace.from_IG02.row_count   205.000000   305.000000   192.000000         False         <NA>
9   1615532400  Tomcat03         trace.self.duration_mean     0.115847     0.178939     0.061365         False         <NA>
10  1615532400  Tomcat03          trace.self.duration_p95     1.004000     1.036775     0.084850         False         <NA>
11  1615532400  Tomcat03             trace.self.row_count  2756.000000  4103.000000  2660.500000         False         <NA>
12  1615532460      IG02         trace.self.duration_mean     0.739148     1.028284     0.414144         False         <NA>
13  1615532460      IG02          trace.self.duration_p95     3.112000     4.404725     2.052650         False         <NA>
14  1615532460      IG02             trace.self.row_count  1921.000000  2435.150000  1703.300000         False         <NA>
15  1615532460      IG02  trace.to_Tomcat03.duration_mean     0.793712     1.159075     0.432528         False         <NA>
16  1615532460      IG02   trace.to_Tomcat03.duration_p95     3.140400     4.788150     2.041900         False         <NA>
17  1615532460  Tomcat03        trace.from_IG02.row_count   229.000000   305.000000   192.000000         False         <NA>
18  1615532460  Tomcat03    trace.from_IG02.duration_mean     0.793712     1.159075     0.432528         False         <NA>
19  1615532460  Tomcat03     trace.from_IG02.duration_p95     3.140400     4.788150     2.041900         False         <NA>```
```

In [ ]:
# In[10]:

# Reuse kernel variables where possible and follow rules
import pandas as pd
import numpy as np

start_ts = 1615532400
end_ts = 1615534200
cmdbs = ["IG01","IG02","MG01","MG02","Tomcat01","Tomcat02","Tomcat03"]

# Attempt to use a row-level joined variable if available
try:
    # trace_window was a per-row joined table created earlier; prefer it
    trace_rows = trace_window
    _ = trace_rows.shape
except NameError:
    # If not available, try to reconstruct from raw trace_df and global trace_agg_all
    try:
        trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
        trace_rows = trace_df.merge(
            trace_agg_all[['cmdb_id', 'trace_name', 'p95', 'p5']],
            on=['cmdb_id', 'trace_name'],
            how='left'
        )
    except Exception as e:
        # If reconstruction fails, create empty DataFrame and proceed to return empty indicator
        trace_rows = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95','p5'])

# Ensure numeric types
trace_rows['timestamp'] = pd.to_numeric(trace_rows['timestamp'], errors='coerce').astype('Int64')
trace_rows['value'] = pd.to_numeric(trace_rows['value'], errors='coerce')

# Compute anomaly flag per row where thresholds exist
has_thresh = trace_rows['p95'].notna() & trace_rows['p5'].notna()
trace_rows['anomaly_flag'] = False
trace_rows.loc[has_thresh, 'anomaly_flag'] = (
    (trace_rows.loc[has_thresh, 'value'] >= trace_rows.loc[has_thresh, 'p95']) |
    (trace_rows.loc[has_thresh, 'value'] <= trace_rows.loc[has_thresh, 'p5'])
)

# Filter by components, time window, and anomaly_flag == True
mask = (
    trace_rows['cmdb_id'].isin(cmdbs) &
    (trace_rows['timestamp'] >= start_ts) &
    (trace_rows['timestamp'] <= end_ts) &
    (trace_rows['anomaly_flag'] == True)
)

selected = trace_rows.loc[mask, ['timestamp','cmdb_id','trace_name','value','p95','p5']].copy()

# If no rows match, create an explicit empty result indicator DataFrame
if selected.empty:
    earliest_trace_anoms = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95','p5','anomaly_type','severity_ratio'])
else:
    # Determine anomaly_type and severity_ratio
    def compute_type_and_severity(row):
        v = row['value']
        p95 = row['p95']
        p5 = row['p5']
        if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
            return pd.Series([pd.NA, pd.NA])
        if not pd.isna(p95) and v >= p95:
            # high
            sev = np.nan if p95 == 0 else float(v) / float(p95)
            return pd.Series(['high', sev])
        if not pd.isna(p5) and v <= p5:
            # low
            sev = np.nan if v == 0 else float(p5) / float(v)
            return pd.Series(['low', sev])
        return pd.Series([pd.NA, pd.NA])

    sel_extra = selected.apply(compute_type_and_severity, axis=1)
    sel_extra.columns = ['anomaly_type', 'severity_ratio']
    selected = pd.concat([selected.reset_index(drop=True), sel_extra.reset_index(drop=True)], axis=1)

    # Sort by timestamp ascending and keep earliest 20 rows
    earliest_trace_anoms = selected.sort_values('timestamp', ascending=True).head(20).reset_index(drop=True)

# Store result in kernel variable and display compactly
earliest_trace_anoms

```
Out[10]:
```
Summary of earliest trace anomalies (stored as earliest_trace_anoms):

- Count & time window: 20 earliest anomaly rows were found between 1615532400 and 1615534200. The earliest anomaly timestamp in this set is 1615532400.

- Components with earliest anomalies: IG01, MG01, MG02, Tomcat03, and Tomcat01.

- Anomaly types and patterns:
  - Both "high" (value >= p95) and "low" (value <= p5) anomalies appear.
  - Many row_count anomalies are near thresholds (small deviations). Examples of "low" row_count cases:
    - IG01 trace.from_root.row_count = 1 (p5 = 1) — low (severity_ratio = 1.0).
    - IG01 trace.to_Tomcat03.row_count = 211 (p5 = 212.8) — low (severity_ratio ≈ 1.0085).
    - Tomcat03 trace.from_IG01.row_count = 211 (p5 = 212.8) — low (severity_ratio ≈ 1.0085).
  - Several duration p95 or duration_mean anomalies are "high" (latency tail exceeded p95), with modest-to-notable severity:
    - Tomcat01 trace.from_root.duration_mean = 2.042s (p95 ≈ 1.233s) — high, severity_ratio ≈ 1.656 (largest in this set).
    - MG02 trace.from_root.duration_mean = 1.048s (p95 ≈ 0.797s) — high, severity_ratio ≈ 1.315.
    - MG01 trace.from_dockerA2.duration_p95 = 1.58s (p95 ≈ 1.409s) — high, severity_ratio ≈ 1.121.
  - Some MG node row_count anomalies show higher-than-threshold values (e.g., MG01/MG02 trace.from_dockerB1 row_count around 295–300 vs p95 ≈ 291–296).

- Severity summary: severity ratios mostly in the range ~1.01–1.66, indicating most anomalies are modest exceedances of thresholds, with Tomcat01's duration being the largest breach among these earliest rows.

- Interpretation & recommended next steps:
  - Early anomalies concentrate on MG nodes (MG01/MG02) and Tomcat services (Tomcat01/Tomcat03), and include both traffic spikes (row_count) and elevated latencies (duration metrics).
  - Correlate these timestamps with metric and log anomalies (use metric_window_anoms and tomcat03_metric_details / tomcat03_trace_details) to determine whether the latency spikes align with CPU/memory, GC, or request-rate changes.
  - Inspect full raw trace rows around the highest-severity timestamps (e.g., Tomcat01 at 1615532460) and check upstream/downstream components to trace propagation.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                         trace_name       value         p95          p5 anomaly_type  severity_ratio
0   1615532400      IG01          trace.from_root.row_count    1.000000    8.650000    1.000000          low        1.000000
1   1615532400      IG01        trace.to_Tomcat03.row_count  211.000000  311.650000  212.800000          low        1.008531
2   1615532400      MG01   trace.from_dockerA1.duration_p95    1.131500    1.118652    0.043445         high        1.011485
3   1615532400      MG01   trace.from_dockerA2.duration_p95    1.580000    1.408955    0.029670         high        1.121398
4   1615532400      MG02   trace.from_dockerA1.duration_p95    1.158550    1.135802    0.030350         high        1.020028
5   1615532400      MG02  trace.from_dockerB1.duration_mean    0.240952    0.230792    0.053058         high        1.044023
6   1615532400      MG02     trace.to_dockerB1.duration_p95    5.864000    5.779040    2.078450         high        1.014701
7   1615532400  Tomcat03          trace.from_IG01.row_count  211.000000  311.650000  212.800000          low        1.008531
8   1615532400  Tomcat03          trace.from_root.row_count   34.000000   31.900000    3.200000         high        1.065831
9   1615532460      MG01      trace.from_dockerB1.row_count  300.000000  291.850000   79.400000         high        1.027925
10  1615532460      MG02      trace.from_root.duration_mean    1.048000    0.797230    0.005800         high        1.314551
11  1615532460      MG02          trace.from_root.row_count    1.000000   34.200000    1.000000          low        1.000000
12  1615532460  Tomcat01      trace.from_root.duration_mean    2.042000    1.233225    0.001015         high        1.655821
13  1615532460  Tomcat01          trace.from_root.row_count    1.000000   21.000000    1.000000          low        1.000000
14  1615532520      MG01      trace.from_dockerB1.row_count  295.000000  291.850000   79.400000         high        1.010793
15  1615532520      MG01        trace.to_dockerA1.row_count  119.000000  233.650000  126.200000          low        1.060504
16  1615532520      MG01        trace.to_dockerB2.row_count  268.000000  268.000000  132.850000         high        1.000000
17  1615532520      MG02      trace.from_dockerB1.row_count  300.000000  296.400000   81.350000         high        1.012146
18  1615532520      MG02        trace.to_dockerA1.row_count  124.000000  235.650000  124.000000          low        1.000000
19  1615532520      MG02        trace.to_dockerA2.row_count  132.000000  231.550000  132.900000          low        1.006818```
```

In [ ]:
# In[11]:

# Reuse kernel variable metric_window_anoms and produce requested filtered rows for Tomcat01
import pandas as pd
import numpy as np

start_ts = 1615532400
end_ts = 1615532520

# Try to reuse existing metric_window_anoms
try:
    _ = metric_window_anoms.shape
except NameError:
    # If not available, create empty DataFrame to avoid failing silently
    metric_window_anoms = pd.DataFrame(columns=[
        'timestamp','cmdb_id','kpi_name','value','p95','p5','anomaly'
    ])

# Ensure numeric types for calculations
metric_window_anoms['timestamp'] = pd.to_numeric(metric_window_anoms['timestamp'], errors='coerce').astype('Int64')
metric_window_anoms['value'] = pd.to_numeric(metric_window_anoms['value'], errors='coerce')
metric_window_anoms['p95'] = pd.to_numeric(metric_window_anoms.get('p95'), errors='coerce')
metric_window_anoms['p5'] = pd.to_numeric(metric_window_anoms.get('p5'), errors='coerce')

# Filter rows for Tomcat01 within the given inclusive window
mask = (
    (metric_window_anoms['cmdb_id'] == 'Tomcat01') &
    (metric_window_anoms['timestamp'] >= start_ts) &
    (metric_window_anoms['timestamp'] <= end_ts)
)
filtered = metric_window_anoms.loc[mask].copy()

# Compute anomaly_flag (based on global thresholds) and anomaly_type/severity_ratio
has_thresh = filtered['p95'].notna() & filtered['p5'].notna()
filtered['anomaly_flag'] = False
filtered.loc[has_thresh, 'anomaly_flag'] = (
    (filtered.loc[has_thresh, 'value'] >= filtered.loc[has_thresh, 'p95']) |
    (filtered.loc[has_thresh, 'value'] <= filtered.loc[has_thresh, 'p5'])
)

def compute_type_and_severity(row):
    v = row['value']
    p95 = row['p95']
    p5 = row['p5']
    if pd.isna(v) or (pd.isna(p95) and pd.isna(p5)):
        return pd.Series([pd.NA, np.nan])
    # high anomaly
    if (not pd.isna(p95)) and (v >= p95):
        severity = np.nan if p95 == 0 else float(v) / float(p95)
        return pd.Series(['high', severity])
    # low anomaly
    if (not pd.isna(p5)) and (v <= p5):
        severity = np.nan if v == 0 else float(p5) / float(v)
        return pd.Series(['low', severity])
    return pd.Series([pd.NA, np.nan])

if not filtered.empty:
    extra = filtered.apply(compute_type_and_severity, axis=1)
    extra.columns = ['anomaly_type', 'severity_ratio']
    filtered = pd.concat([filtered.reset_index(drop=True), extra.reset_index(drop=True)], axis=1)
else:
    # create the expected columns if empty
    filtered['anomaly_type'] = pd.Series(dtype='object')
    filtered['severity_ratio'] = pd.Series(dtype='float')

# Select requested columns in order
tomcat01_metric_anom_rows = filtered[
    ['timestamp','cmdb_id','kpi_name','value','p95','p5','anomaly_flag','anomaly_type','severity_ratio']
].copy()

# Sort by timestamp ascending then severity_ratio descending (NaNs last), limit to top 20
tomcat01_metric_anom_rows = tomcat01_metric_anom_rows.sort_values(
    by=['timestamp','severity_ratio'],
    ascending=[True, False],
    na_position='last'
).head(20).reset_index(drop=True)

# Display the resulting compact table (stored in kernel variable tomcat01_metric_anom_rows)
tomcat01_metric_anom_rows

```
Out[11]:
```
Summary of Tomcat01 metric rows (timestamp 1615532400–1615532520; stored as tomcat01_metric_anom_rows):

- Scope & count: Returned up to 20 rows sorted by timestamp ascending then severity_ratio desc. Most rows are at 1615532400; one row at 1615532460.

- Columns returned: timestamp, cmdb_id, kpi_name, value, p95, p5, anomaly_flag, anomaly_type, severity_ratio.

- Key observations:
  - Many Tomcat request/session/thread/memory metrics at 1615532400 are flagged anomalous (anomaly_flag = True).
  - Examples (selected):
    - Tomcat-Requests..."Processing..." (kpi truncated): value ≈ 1.517e9, p95 ≈ 1.5265e9, p5 ≈ 1.517871e9 → anomaly_flag = True, anomaly_type = low, severity_ratio ≈ 1.00038 (value is slightly below p5).
    - Tomcat-Requests..."RequestCount": value ≈ 2.2948e6, p95 ≈ 2.3091e6, p5 ≈ 2.2955e6 → anomaly_flag = True, anomaly_type = low, severity_ratio ≈ 1.00032.
    - OSLinux MEMORY MEMUsedMemPerc = 98 (value == p95 == p5) → anomaly_flag = True, anomaly_type = high, severity_ratio = 1.0.
    - Request ErrorCount = 987 (value == p95 == p5) → anomaly_flag = True, anomaly_type = high, severity_ratio = 1.0.
    - Thread metrics CurrentThreads=100, MaxThreads=500 — constant and flagged high (severity_ratio = 1.0).
    - Several Tomcat session metrics are constant 0 and flagged anomalous (anomaly_type reported as high). These rows show NaN severity_ratio because p95/p5 are 0, causing division-by-zero when computing ratio.
  - Non-anomalous examples: some single-CPU util metrics at 1615532400 are not anomalous (anomaly_flag = False).
  - At 1615532460 an OSLinux local disk read metric is flagged low (value=112, p5=146.2) with severity_ratio ≈ 1.305 (largest low-severity in this subset).

- Notes on severity_ratio and NaNs:
  - severity_ratio = value/p95 for 'high' anomalies; = p5/value for 'low' anomalies.
  - Rows where p95 or value are zero produce NaN severity_ratio (observed for session metrics with p95==0 and value==0). Interpret those carefully — NaN indicates division not meaningful, not absence of anomaly.

- Interpretation / next steps:
  - Many anomalies are caused by constant metric values equal to global thresholds (p95/p5 equal to observed values), producing full-window flags — these may be threshold artifacts rather than true operational issues.
  - The low anomalies on request counts and processing metrics are very slight breaches (severity ratios ≈ 1.0003–1.0004).
  - The disk read low at 1615532460 (severity ≈ 1.305) and any non-trivial duration/throughput breaches merit closer inspection.
  - Recommended: inspect the raw time series around the timestamps in tomcat01_metric_anom_rows and correlate with trace anomalies and logs for causality.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                                           kpi_name         value           p95            p5  anomaly_flag anomaly_type  severity_ratio
0   1615532400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_Processin...  1.517294e+09  1.526537e+09  1.517871e+09          True          low        1.000380
1   1615532400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_RequestCo...  2.294778e+06  2.309119e+06  2.295511e+06          True          low        1.000319
2   1615532400  Tomcat01       OSLinux-OSLinux_MEMORY_MEMORY_MEMUsedMemPerc  9.800000e+01  9.800000e+01  9.800000e+01          True         high        1.000000
3   1615532400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_ErrorCoun...  9.870000e+02  9.870000e+02  9.870000e+02          True         high        1.000000
4   1615532400  Tomcat01  Tomcat-Requests_7441-"http-nio-8003"_MaxTimeRe...  4.053960e+05  4.053960e+05  4.053960e+05          True         high        1.000000
5   1615532400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  1.000000e+02  1.000000e+02  1.000000e+02          True         high        1.000000
6   1615532400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_MaxThreads...  5.000000e+02  5.000000e+02  5.000000e+02          True         high        1.000000
7   1615532400  Tomcat01                    OSLinux-CPU_CPU-1_SingleCpuUtil  4.662800e+00  8.269970e+01  3.865705e+00         False         <NA>             NaN
8   1615532400  Tomcat01                    OSLinux-CPU_CPU-3_SingleCpuUtil  2.662700e+00  6.993390e+00  2.312765e+00         False         <NA>             NaN
9   1615532400  Tomcat01    Tomcat-Sessions_7441--UOCP_SESSIONActiveCounter  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
10  1615532400  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONKeepaliveCou...  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
11  1615532400  Tomcat01  Tomcat-Sessions_7441--UOCP_SESSIONRejectedSess...  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
12  1615532400  Tomcat01        Tomcat-Sessions_7441--_SESSIONActiveCounter  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
13  1615532400  Tomcat01     Tomcat-Sessions_7441--_SESSIONKeepaliveCounter  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
14  1615532400  Tomcat01     Tomcat-Sessions_7441--_SESSIONRejectedSessions  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
15  1615532400  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
16  1615532400  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
17  1615532400  Tomcat01  Tomcat-Sessions_7441--logHome_IS_UNDEFINED_SES...  0.000000e+00  0.000000e+00  0.000000e+00          True         high             NaN
18  1615532400  Tomcat01  Tomcat-Threads_7441-"http-nio-8003"_CurrentThr...  8.000000e+00  1.550000e+01  2.000000e+00         False         <NA>             NaN
19  1615532460  Tomcat01  OSLinux-OSLinux_LOCALDISK_LOCALDISK-sdb_DSKRea...  1.120000e+02  2.526000e+02  1.462000e+02          True          low        1.305357```
```

In [ ]:
# In[12]:

# Reuse existing kernel variables where possible to extract row-level trace anomalies for Tomcat01
import pandas as pd
import numpy as np

start_ts = 1615532400
end_ts = 1615534200

# Prefer existing per-row joined variable 'trace_window' if present; otherwise reconstruct from trace_df + trace_agg_all
try:
    trace_rows = trace_window  # previously created per-row joined DataFrame
    _ = trace_rows.shape
except NameError:
    # Reconstruct from raw trace_df and global thresholds trace_agg_all
    trace_df['value'] = pd.to_numeric(trace_df['value'], errors='coerce')
    trace_rows = trace_df.merge(
        trace_agg_all[['cmdb_id', 'trace_name', 'p95', 'p5']],
        on=['cmdb_id', 'trace_name'],
        how='left'
    )

# Ensure numeric types
trace_rows['timestamp'] = pd.to_numeric(trace_rows['timestamp'], errors='coerce').astype('Int64')
trace_rows['value'] = pd.to_numeric(trace_rows['value'], errors='coerce')
trace_rows['p95'] = pd.to_numeric(trace_rows.get('p95'), errors='coerce')
trace_rows['p5'] = pd.to_numeric(trace_rows.get('p5'), errors='coerce')

# Compute anomaly_flag where thresholds exist
has_thresh = trace_rows['p95'].notna() & trace_rows['p5'].notna()
trace_rows['anomaly_flag'] = False
trace_rows.loc[has_thresh, 'anomaly_flag'] = (
    (trace_rows.loc[has_thresh, 'value'] >= trace_rows.loc[has_thresh, 'p95']) |
    (trace_rows.loc[has_thresh, 'value'] <= trace_rows.loc[has_thresh, 'p5'])
)

# Filter for Tomcat01 within the window and anomaly_flag == True
mask = (
    (trace_rows['cmdb_id'] == 'Tomcat01') &
    (trace_rows['timestamp'] >= start_ts) &
    (trace_rows['timestamp'] <= end_ts) &
    (trace_rows['anomaly_flag'] == True)
)
sel = trace_rows.loc[mask, ['timestamp','cmdb_id','trace_name','value','p95','p5']].copy()

# If no matching rows, produce empty DataFrame with expected columns
if sel.empty:
    tomcat01_trace_anom_rows = pd.DataFrame(columns=['timestamp','cmdb_id','trace_name','value','p95','p5','anomaly_type','severity_ratio'])
else:
    # Determine anomaly_type and severity_ratio
    def type_and_sev(row):
        v = row['value']
        p95 = row['p95']
        p5 = row['p5']
        if pd.isna(v):
            return pd.Series([pd.NA, np.nan])
        if (not pd.isna(p95)) and (v >= p95):
            sev = np.nan if p95 == 0 else float(v) / float(p95)
            return pd.Series(['high', sev])
        if (not pd.isna(p5)) and (v <= p5):
            sev = np.nan if v == 0 else float(p5) / float(v)
            return pd.Series(['low', sev])
        return pd.Series([pd.NA, np.nan])

    extra = sel.apply(type_and_sev, axis=1)
    extra.columns = ['anomaly_type','severity_ratio']
    tomcat01_trace_anom_rows = pd.concat([sel.reset_index(drop=True), extra.reset_index(drop=True)], axis=1)

    # Sort by timestamp asc then severity_ratio desc (NaNs last), limit to top 20
    tomcat01_trace_anom_rows = tomcat01_trace_anom_rows.sort_values(
        by=['timestamp','severity_ratio'], ascending=[True, False], na_position='last'
    ).head(20).reset_index(drop=True)

# Store result in kernel variable and display the compact table
tomcat01_trace_anom_rows

```
Out[12]:
```
Summary of Tomcat01 trace anomalies (stored as tomcat01_trace_anom_rows):

- Scope & storage:
  - Returned 20 earliest trace-anomaly rows for Tomcat01 in the window 1615532400–1615534200.
  - Results are stored in kernel variable tomcat01_trace_anom_rows.

- Time range of these anomalies:
  - Earliest anomalies at 1615532460, with a cluster of stronger anomalies at 1615532700 and a larger cluster at 1615532820.

- Most severe early anomaly:
  - 1615532460 — trace.from_root.duration_mean = 2.042s vs p95=1.233s → high, severity_ratio ≈ 1.656 (largest breach in this set).

- Other notable anomalies:
  - 1615532460 — trace.from_root.row_count = 1 (low vs p5=1) — trivial low.
  - 1615532580 — trace.from_root.duration_p95 = 2.574s vs p95=2.281s → high, severity ≈ 1.128.
  - 1615532640 & 1615532700 — elevated outgoing row_count to MG01 (316, 334) and very large self.row_count (4455 at 1615532700) — all high, severity ratios ~1.02–1.08.
  - 1615532820 — many high-duration anomalies (to_MG01.duration_p95, from_IG02.duration_p95, to_MG02.duration_p95, etc.) with severity ratios ~1.15–1.29, and several duration_mean and self.duration_mean anomalies (ratios ~1.08–1.12). Also a couple of low-duration anomalies (very small values below p5) with severity_ratios >1 (p5/value).
  - Several row_count low anomalies (values slightly below p5), severity ratios ~1.00–1.01 (small deviations).

- Pattern and interpretation:
  - Two primary problems visible:
    1. Latency spikes: multiple duration metrics exceed their p95 (Tomcat→MG and from IG paths), especially around 1615532820 and earlier at 1615532460/1615532580.
    2. Traffic spikes: large increases in trace.self.row_count and to/MG row_count (e.g., 4455 self row_count at 1615532700).
  - Severity is modest-to-moderate (most ratios between ~1.02 and ~1.30), with one larger breach at ~1.656.
  - Both upstream (IG) and downstream (MG) interactions with Tomcat01 show anomalies, suggesting either traffic surge or processing/latency issues inside Tomcat01 or its dependencies.

- Recommended next steps:
  - Correlate these timestamps (notably 1615532460, ~1615532700, and ~1615532820) with Tomcat01 metric anomalies (CPU, memory, GC, disk I/O) and logs to identify resource pressure or errors.
  - Inspect raw traces around those timestamps to see failed/slow spans and whether latency originates in Tomcat01 or its downstream MG services.

The original code execution output of IPython Kernel is also provided below for reference:

timestamp   cmdb_id                     trace_name        value          p95           p5 anomaly_type  severity_ratio
0   1615532460  Tomcat01  trace.from_root.duration_mean     2.042000     1.233225     0.001015         high        1.655821
1   1615532460  Tomcat01      trace.from_root.row_count     1.000000    21.000000     1.000000          low        1.000000
2   1615532580  Tomcat01   trace.from_root.duration_p95     2.574000     2.281400     0.005553         high        1.128255
3   1615532640  Tomcat01        trace.to_MG01.row_count   316.000000   310.150000   180.700000         high        1.018862
4   1615532700  Tomcat01        trace.to_MG01.row_count   334.000000   310.150000   180.700000         high        1.076898
5   1615532700  Tomcat01           trace.self.row_count  4455.000000  4215.850000  2721.400000         high        1.056726
6   1615532700  Tomcat01      trace.from_IG02.row_count   326.000000   315.650000   203.150000         high        1.032789
7   1615532700  Tomcat01      trace.from_IG01.row_count   300.000000   297.750000   199.900000         high        1.007557
8   1615532820  Tomcat01   trace.from_root.duration_p95     0.003600     2.281400     0.005553          low        1.542361
9   1615532820  Tomcat01  trace.from_root.duration_mean     0.000667     1.233225     0.001015          low        1.521964
10  1615532820  Tomcat01     trace.to_MG01.duration_p95     5.565000     4.331247     1.161420         high        1.284849
11  1615532820  Tomcat01   trace.from_IG02.duration_p95     5.490500     4.320733     2.056855         high        1.270734
12  1615532820  Tomcat01     trace.to_MG02.duration_p95     4.733000     4.110290     1.118320         high        1.151500
13  1615532820  Tomcat01    trace.to_MG01.duration_mean     1.021687     0.915190     0.285824         high        1.116366
14  1615532820  Tomcat01  trace.from_IG02.duration_mean     1.119131     1.031557     0.402532         high        1.084895
15  1615532820  Tomcat01       trace.self.duration_mean     0.159456     0.147745     0.058085         high        1.079263
16  1615532820  Tomcat01   trace.from_IG01.duration_p95     4.787000     4.520330     2.056660         high        1.058993
17  1615532820  Tomcat01    trace.to_MG02.duration_mean     0.908315     0.858353     0.292983         high        1.058207
18  1615532820  Tomcat01  trace.from_IG01.duration_mean     0.996648     0.961304     0.412264         high        1.036767
19  1615532820  Tomcat01      trace.from_IG01.row_count   199.000000   297.750000   199.900000          low        1.004523```
```